# POSEIDON + LIFE: emisión térmica de escenarios ExoFarm

## Superficie terrestre mixta y nubes parcheadas

Este notebook genera espectros térmicos absolutos de los escenarios A0–A3 mediante tres configuraciones:

1. **Control gris y despejado:** superficie negra, sin estructura espectral superficial.
2. **Superficie mixta tipo Tierra y despejada:**  
   \(70\%\) océano, \(15\%\) bosque, \(5\%\) nieve/hielo y \(10\%\) arena.
3. **Superficie mixta + nubes parcheadas:** la misma distribución superficial con un benchmark de \(50\%\) de cobertura nubosa.

Los perfiles \(P\)-\(T\) y químicos se leen con la misma convención empleada en `exofarm_transmission_workflow.py`:

- `read_PT_file(..., P_column=2, T_column=3, skiprows=1)`
- `read_chem_file(..., chem_species_file=CHEM_SPECIES_FILE, skiprows=1)`
- archivos `Trappist_A0_PreAgri`, `Trappist_A1_Current`, `Trappist_A2_Moderate` y `Trappist_A3_Extreme`.

La salida es:

\[
F_p(\lambda),
\]

el flujo absoluto recibido por el observador, obtenido con:

```python
object_type="directly_imaged"
spectrum_type="direct_emission"
```

No se realizan retrievals ni simulaciones instrumentales en este notebook.

## Diseño experimental

Las tres configuraciones comparten exactamente:

- planeta, gravedad y distancia;
- perfiles \(T(P)\) y \(X_i(P)\);
- presión superficial;
- opacidades moleculares;
- malla espectral.

Así se separan tres efectos:

\[
\Delta F_{\rm superficie}
=
F_{\rm mixta,clear}-F_{\rm gray,clear},
\]

\[
\Delta F_{\rm nube}
=
F_{\rm mixta,cloudy}-F_{\rm mixta,clear},
\]

\[
\Delta F_{\rm ExoFarm}
=
F_{A_j}-F_{A0}.
\]

La distribución de superficie y el benchmark nuboso siguen el ejemplo oficial de POSEIDON para una Tierra sencilla. Sin embargo, el ejemplo oficial fue construido para luz reflejada. Este notebook **verifica explícitamente** que los espectros superficiales instalados cubran todo el rango MIR solicitado; no extrapola silenciosamente datos visibles/NIR hasta 18.5 μm.

## 0. Requisitos

Se requiere:

- POSEIDON con la base `Temperate`;
- base de aerosoles Mie actualizada;
- carpeta `surface_reflectivities`;
- archivos de perfiles ExoFarm en una carpeta `profiles`.

Documentación oficial relevante:

- [Rocky planets with reflecting and emitting surfaces](https://poseidon-retrievals.readthedocs.io/en/latest/content/notebooks/reflection_emission_surfaces.html)
- [Modern Earth surface distribution and patchy clouds](https://poseidon-retrievals.readthedocs.io/en/latest/content/notebooks/reflection_hwo.html)
- [Direct emission of directly imaged objects](https://poseidon-retrievals.readthedocs.io/en/latest/content/notebooks/brown_dwarf.html)

In [ ]:
from pathlib import Path
import importlib.metadata
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.constants import c, parsec as pc
from IPython.display import display

warnings.filterwarnings("default")
np.set_printoptions(precision=5, suppress=True)

try:
    import POSEIDON
    from POSEIDON.constants import M_E, R_E
    from POSEIDON.core import (
        check_atmosphere_physical,
        compute_spectrum,
        create_planet,
        define_model,
        make_atmosphere,
        read_opacities,
        wl_grid_constant_R,
    )
    from POSEIDON.utility import bin_spectrum, read_chem_file, read_PT_file
    from POSEIDON.surfaces import (
        interpolate_surface_components,
        load_surface_components,
    )
except Exception as exc:
    raise RuntimeError(
        "No fue posible importar POSEIDON o alguno de sus módulos. "
        "Verifica la instalación y la variable de entorno de los inputs."
    ) from exc

try:
    POSEIDON_VERSION = importlib.metadata.version("POSEIDON")
except importlib.metadata.PackageNotFoundError:
    POSEIDON_VERSION = getattr(
        POSEIDON,
        "__version__",
        "instalación desde código fuente",
    )

print("POSEIDON:", POSEIDON_VERSION)
print("Directorio actual:", Path.cwd())

## 1. Configuración editable

La configuración planetaria conserva inicialmente TRAPPIST-1e, porque los nombres y perfiles del workflow suministrado corresponden a ese planeta. La distancia es necesaria para `direct_emission`.

La malla nativa se fija en \(R=1000\). Para exploración visual se rebinea posteriormente a \(R=300,100,50\).

In [ ]:
# ------------------------------------------------------------------
# Directorios
# ------------------------------------------------------------------
# Define una ruta explícita aquí si la detección automática no encuentra
# la carpeta correcta.
PROFILES_DIR_OVERRIDE = None

def locate_profiles_dir():
    if PROFILES_DIR_OVERRIDE is not None:
        candidate = Path(PROFILES_DIR_OVERRIDE).expanduser().resolve()
        if not candidate.exists():
            raise FileNotFoundError(f"No existe PROFILES_DIR_OVERRIDE: {candidate}")
        return candidate

    candidates = [
        Path.cwd() / "profiles",
        Path.cwd().parent / "profiles",
        Path.cwd().parent.parent / "profiles",
        Path.cwd() / "Transmission_Spectroscopy" / "profiles",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    expected = "\n".join(f"  - {candidate.resolve()}" for candidate in candidates)
    raise FileNotFoundError(
        "No encontré la carpeta de perfiles. Probé:\n"
        f"{expected}\n"
        "Asigna PROFILES_DIR_OVERRIDE manualmente."
    )

PROFILES_DIR = locate_profiles_dir()
OUTPUT_DIR = Path.cwd() / "POSEIDON_output" / "thermal_direct_emission"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Planeta
# ------------------------------------------------------------------
PLANET_NAME = "TRAPPIST-1e"
RADIUS_EARTH = 0.920
MASS_EARTH = 0.6356
DISTANCE_PC = 12.1
T_EQ_K = 255.0

# ------------------------------------------------------------------
# Atmósfera
# ------------------------------------------------------------------
P_SURF_BAR = 1.0
P_REF_BAR = P_SURF_BAR
P_TOP_BAR = 1.0e-6
P_GRID_SPLIT_BAR = 1.0e-2
P_BOTTOM_BAR = 1.0e1
N_UPPER = 50
N_LOWER = 150

# ------------------------------------------------------------------
# Espectro LIFE
# ------------------------------------------------------------------
WL_MIN_UM = 4.0
WL_MAX_UM = 18.5
R_NATIVE = 1000
R_PLOT = 100
R_LIFE_EXPLORATORY = 50
USE_PHOTOSPHERE_RADIUS = True

# ------------------------------------------------------------------
# Ejecución
# ------------------------------------------------------------------
RUN_GRAY_CONTROL = True
RUN_CLEAR_MIXED_SURFACE = True
RUN_CLOUDY_STAGE = True
RUN_LEAVE_ONE_OUT = True

# Si es True, el notebook se detiene cuando alguna superficie no cubre
# 4–18.5 µm. No se recomienda extrapolar reflectancias arbitrariamente.
STRICT_SURFACE_COVERAGE = True

print("Profiles:", PROFILES_DIR)
print("Outputs :", OUTPUT_DIR)

## 2. Escenarios y formato de archivos

Este bloque reproduce los identificadores y el orden químico de `exofarm_transmission_workflow.py`.

`CHEM_SPECIES_FILE` debe permanecer sincronizado con el formato de exportación de VULCAN. POSEIDON seleccionará solamente las especies requeridas por el modelo.

In [ ]:
SCENARIOS = {
    "A0": {
        "label": "Trappist_A0_PreAgri",
        "name": "Pre-agricultural",
    },
    "A1": {
        "label": "Trappist_A1_Current",
        "name": "Current Earth",
    },
    "A2": {
        "label": "Trappist_A2_Moderate",
        "name": "Moderate ExoFarm",
    },
    "A3": {
        "label": "Trappist_A3_Extreme",
        "name": "Extreme ExoFarm",
    },
}
SCENARIO_KEYS = tuple(SCENARIOS)

BULK_SPECIES = ["N2"]
PARAM_SPECIES = [
    "H2O",
    "CO2",
    "CH4",
    "O2",
    "O3",
    "N2O",
    "NH3",
    "CO",
]

CHEM_SPECIES_FILE = [
    "OH", "H2", "H2O", "H", "O", "CH", "C", "CH2", "CH3", "CH4",
    "C2", "C2H2", "C2H", "C2H3", "C2H4", "C2H5", "C2H6",
    "CO", "CO2", "CH2OH", "H2CO", "HCO", "CH3O", "CH3OH", "CH3CO",
    "O2", "H2CCO", "HCCO", "CH3O2", "HO2", "CH3OOH",
    "N", "NH", "CN", "HCN", "NH2", "N2", "NH3", "NO",
    "N2H2", "N2H", "N2H3", "N2H4", "HNO", "H2CN", "HNCO", "NO2", "N2O",
    "C4H2", "CH2NH2", "CH2NH", "CH3NH2", "CH3CHO",
    "O3", "NO3", "HNO3", "HNO2", "NCO", "N2O5",
    "S", "SH", "S2", "SO", "H2S", "CS", "COS", "CS2", "NS", "HS2", "SO2",
    "S4", "S8", "HCS", "S3", "H2O2", "SO3", "HSO3", "HSO", "H2SO4",
    "HC3N", "CH3CN", "CH2CN", "C2H3CN", "CH3SH", "CH3S",
    "C3H3", "C3H2", "C3H4", "C6H5", "C6H6", "C4H3", "C4H5",
    "S2O", "O_1", "CH2_1", "N_2D", "He", "H2O_l_s", "H2SO4_l",
]

expected_files = []
for key, info in SCENARIOS.items():
    expected_files.extend([
        PROFILES_DIR / f"{info['label']}_PT.txt",
        PROFILES_DIR / f"{info['label']}_chem.txt",
    ])

missing = [path for path in expected_files if not path.exists()]
if missing:
    print("Archivos faltantes:")
    for path in missing:
        print("  ", path)
    raise FileNotFoundError(
        "Faltan perfiles A0–A3. Revisa PROFILES_DIR y los nombres de archivo."
    )

print("Se encontraron los 8 archivos de perfiles.")

## 3. Mallas y planeta

Para resolver una nube delgada cerca de la superficie se usa una malla más fina entre \(10\) y \(10^{-2}\) bar, siguiendo la recomendación del tutorial terrestre de POSEIDON.

In [ ]:
P_upper = np.logspace(
    np.log10(P_TOP_BAR),
    np.log10(P_GRID_SPLIT_BAR),
    N_UPPER,
)
P_lower = np.logspace(
    np.log10(P_GRID_SPLIT_BAR),
    np.log10(P_BOTTOM_BAR),
    N_LOWER,
)

P = np.unique(np.concatenate([P_upper, P_lower]))[::-1]
wl = wl_grid_constant_R(WL_MIN_UM, WL_MAX_UM, R_NATIVE)

R_p = RADIUS_EARTH * R_E
M_p = MASS_EARTH * M_E
distance_m = DISTANCE_PC * pc

planet = create_planet(
    PLANET_NAME,
    R_p,
    mass=M_p,
    T_eq=T_EQ_K,
    d=distance_m,
)

# En direct_emission no se divide por un espectro estelar.
star = None

print("Capas de presión:", len(P))
print("Rango P [bar]:", P.min(), "–", P.max())
print("Puntos espectrales:", len(wl))
print("Rango λ [µm]:", wl.min(), "–", wl.max())

In [ ]:
plt.figure(figsize=(6.8, 4.5))
plt.plot(np.arange(len(P)), P)
plt.yscale("log")
plt.gca().invert_yaxis()
plt.xlabel("Índice de capa")
plt.ylabel("Presión [bar]")
plt.title("Malla de presión refinada cerca de la superficie")
plt.grid(True, which="both", alpha=0.25)
plt.show()

## 4. Superficie tipo Tierra

Se adopta exactamente la distribución del ejemplo sencillo de POSEIDON:

\[
f_{\rm ocean}=0.70,\quad
f_{\rm forest}=0.15,\quad
f_{\rm snow}=0.05,\quad
f_{\rm sand}=0.10.
\]

Los porcentajes se aplican a los albedos antes de la transferencia radiativa, lo que requiere un solo cálculo por atmósfera. En emisión térmica POSEIDON usa:

\[
\epsilon_\lambda = 1-A_\lambda.
\]

In [ ]:
SURFACE_COMPONENTS = [
    "USGS_ocean_seawater_GG25",
    "USGS_aspen_Z26",
    "Grenfell1994_antarctic_Z26",
    "USGS_quartz_sand_GG25",
]

SURFACE_FRACTIONS = np.array([0.70, 0.15, 0.05, 0.10], dtype=float)
SURFACE_FRACTIONS /= SURFACE_FRACTIONS.sum()

SURFACE_LABELS = [
    "Ocean",
    "Forest / aspen",
    "Snow / ice",
    "Quartz sand",
]

surface_data = load_surface_components(SURFACE_COMPONENTS)

coverage_rows = []
for name, data in zip(SURFACE_COMPONENTS, surface_data):
    array = np.asarray(data)
    coverage_rows.append({
        "component": name,
        "wl_min_um": float(np.nanmin(array[:, 0])),
        "wl_max_um": float(np.nanmax(array[:, 0])),
        "covers_requested_range": bool(
            np.nanmin(array[:, 0]) <= WL_MIN_UM
            and np.nanmax(array[:, 0]) >= WL_MAX_UM
        ),
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

coverage_ok = bool(coverage_df["covers_requested_range"].all())
if not coverage_ok:
    message = (
        "Una o más superficies no cubren el rango 4–18.5 µm. "
        "El ejemplo HWO fue diseñado para reflexión visible/NIR. "
        "Para un cálculo MIR físicamente defendible, sustituye los archivos "
        "por reflectancias/emisividades de laboratorio que cubran todo el rango."
    )
    if STRICT_SURFACE_COVERAGE:
        raise ValueError(message)
    warnings.warn(message)

surface_albedos = interpolate_surface_components(
    wl,
    SURFACE_COMPONENTS,
    surface_data,
)

surface_albedos = np.asarray(surface_albedos)
combined_albedo = np.sum(
    SURFACE_FRACTIONS[:, None] * surface_albedos,
    axis=0,
)
combined_emissivity = 1.0 - combined_albedo

if np.any((combined_albedo < 0.0) | (combined_albedo > 1.0)):
    raise ValueError("La mezcla superficial produjo albedos fuera de [0,1].")

In [ ]:
plt.figure(figsize=(10, 5))
for index, label in enumerate(SURFACE_LABELS):
    plt.plot(
        wl,
        1.0 - surface_albedos[index],
        label=f"{label} ({100 * SURFACE_FRACTIONS[index]:.0f}%)",
        alpha=0.75,
    )

plt.plot(
    wl,
    combined_emissivity,
    linewidth=2.7,
    label="Mixture",
)
plt.xlabel("Longitud de onda [µm]")
plt.ylabel("Emisividad superficial")
plt.title("Emisividades de la superficie tipo Tierra")
plt.ylim(0.0, 1.05)
plt.grid(True, alpha=0.25)
plt.legend(ncol=2)
plt.show()

## 5. Definición de los tres modelos

### Modelo G — gris, despejado

Control para aislar las firmas atmosféricas.

### Modelo S — superficie mixta, despejado

Modelo principal de la primera etapa.

### Modelo C — superficie mixta, nubes parcheadas

Segunda etapa. Usa un slab Mie y `thermal_scattering=True`.

El aerosol `H2O` reproduce el benchmark oficial de hielo de agua. Si se desea una nube baja líquida, debe verificarse la etiqueta `H2O_l` disponible en la instalación y ajustarse el tamaño de partícula.

In [ ]:
common_model_kwargs = dict(
    bulk_species=BULK_SPECIES,
    param_species=PARAM_SPECIES,
    object_type="directly_imaged",
    PT_profile="file_read",
    X_profile="file_read",
    radius_unit="R_E",
    surface=True,
    reflection=False,
    thermal=True,
)

model_gray = define_model(
    "ExoFarm_direct_gray_clear",
    surface_model="gray",
    thermal_scattering=False,
    **common_model_kwargs,
)

model_surface_clear = define_model(
    "ExoFarm_direct_EarthSurface_clear",
    surface_model="lab_data",
    surface_components=SURFACE_COMPONENTS,
    surface_percentage_option="linear",
    surface_percentage_apply_to="albedos",
    thermal_scattering=False,
    **common_model_kwargs,
)

CLOUD_AEROSOL = "H2O"

model_surface_cloudy = define_model(
    "ExoFarm_direct_EarthSurface_patchyCloud",
    surface_model="lab_data",
    surface_components=SURFACE_COMPONENTS,
    surface_percentage_option="linear",
    surface_percentage_apply_to="albedos",
    cloud_model="Mie",
    cloud_type="slab",
    aerosol_species=[CLOUD_AEROSOL],
    cloud_dim=2,
    thermal_scattering=True,
    **common_model_kwargs,
)

print("Gray parameters:")
print(model_gray["param_names"])
print()
print("Mixed clear parameters:")
print(model_surface_clear["param_names"])
print()
print("Mixed cloudy parameters:")
print(model_surface_cloudy["param_names"])
print()
print("Cloud parameter order:")
print(model_surface_cloudy.get("cloud_param_names"))

## 6. Carga de perfiles \(P-T\) y químicos

Este bloque sigue el workflow suministrado. Todos los escenarios se interpolan sobre la misma malla `P`.

In [ ]:
temperatures = {}
compositions = {}

chemical_species_reference = model_surface_clear["chemical_species"]

for scenario_key, scenario in SCENARIOS.items():
    label = scenario["label"]

    temperatures[scenario_key] = read_PT_file(
        str(PROFILES_DIR),
        f"{label}_PT.txt",
        P,
        skiprows=1,
        P_column=2,
        T_column=3,
    )

    compositions[scenario_key] = read_chem_file(
        str(PROFILES_DIR),
        f"{label}_chem.txt",
        P,
        CHEM_SPECIES_FILE,
        chem_species_in_model=chemical_species_reference,
        skiprows=1,
    )

    T = temperatures[scenario_key]
    X = compositions[scenario_key]

    if T.shape != P.shape:
        raise ValueError(
            f"{scenario_key}: T tiene forma {T.shape}, se esperaba {P.shape}."
        )

    expected_X_shape = (len(chemical_species_reference), len(P))
    if X.shape != expected_X_shape:
        raise ValueError(
            f"{scenario_key}: X tiene forma {X.shape}, "
            f"se esperaba {expected_X_shape}."
        )

    if np.any(~np.isfinite(T)) or np.any(~np.isfinite(X)):
        raise ValueError(f"{scenario_key}: perfiles con NaN o infinito.")

    if np.any(X < 0):
        raise ValueError(f"{scenario_key}: razones de mezcla negativas.")

print("Orden químico usado por POSEIDON:")
print(chemical_species_reference)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for scenario_key, scenario in SCENARIOS.items():
    axes[0].plot(
        temperatures[scenario_key],
        P,
        label=scenario_key,
    )

axes[0].set_yscale("log")
axes[0].invert_yaxis()
axes[0].set_xlabel("Temperatura [K]")
axes[0].set_ylabel("Presión [bar]")
axes[0].set_title("Perfiles P–T")
axes[0].grid(True, which="both", alpha=0.25)
axes[0].legend()

species_to_plot = ["H2O", "CH4", "O3", "N2O", "NH3", "CO2"]
for species in species_to_plot:
    if species not in chemical_species_reference:
        continue
    idx = list(chemical_species_reference).index(species)
    axes[1].plot(
        np.maximum(compositions["A3"][idx], 1.0e-30),
        P,
        label=species,
    )

axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].invert_yaxis()
axes[1].set_xlabel("Razón de mezcla")
axes[1].set_ylabel("Presión [bar]")
axes[1].set_title("Composición A3")
axes[1].grid(True, which="both", alpha=0.25)
axes[1].legend(ncol=2)

fig.tight_layout()
plt.show()

## 7. Parámetros de superficie y nubes

Para la superficie mixta:

```text
[log_P_surf, ocean, forest, snow, sand]
```

Para la nube parcheada del benchmark:

- cobertura: 0.5;
- cima: 0.5 bar;
- base: 0.6 bar;
- radio medio: 1 μm;
- `log_X_cloud = -3`.

Los porcentajes superficiales y nubosos se mantienen idénticos en A0–A3; solo cambia la atmósfera suministrada por VULCAN.

In [ ]:
LOG_P_SURF = np.log10(P_SURF_BAR)

surface_params_gray = np.array([LOG_P_SURF], dtype=float)

surface_params_mixed = np.array(
    [
        LOG_P_SURF,
        SURFACE_FRACTIONS[0],
        SURFACE_FRACTIONS[1],
        SURFACE_FRACTIONS[2],
        SURFACE_FRACTIONS[3],
    ],
    dtype=float,
)

F_CLOUD = 0.50
P_CLOUD_TOP_BAR = 0.50
P_CLOUD_BOTTOM_BAR = 0.60
R_CLOUD_UM = 1.0
LOG_X_CLOUD = -3.0

cloud_params = np.array(
    [
        F_CLOUD,
        np.log10(P_CLOUD_TOP_BAR),
        np.log10(P_CLOUD_BOTTOM_BAR) - np.log10(P_CLOUD_TOP_BAR),
        np.log10(R_CLOUD_UM),
        LOG_X_CLOUD,
    ],
    dtype=float,
)

print("surface_params_mixed:", surface_params_mixed)
print("cloud_params:", cloud_params)

## 8. Construcción de atmósferas

El perfil térmico determina la temperatura de la superficie en \(P_{\rm surf}\).

In [ ]:
atmospheres_gray = {}
atmospheres_surface_clear = {}
atmospheres_surface_cloudy = {}

for scenario_key in SCENARIO_KEYS:
    T_input = temperatures[scenario_key]
    X_input = compositions[scenario_key]

    if RUN_GRAY_CONTROL:
        atmospheres_gray[scenario_key] = make_atmosphere(
            planet,
            model_gray,
            P,
            P_REF_BAR,
            R_p,
            T_input=T_input,
            X_input=X_input,
            surface_params=surface_params_gray,
        )

    if RUN_CLEAR_MIXED_SURFACE:
        atmospheres_surface_clear[scenario_key] = make_atmosphere(
            planet,
            model_surface_clear,
            P,
            P_REF_BAR,
            R_p,
            T_input=T_input,
            X_input=X_input,
            surface_params=surface_params_mixed,
        )

    if RUN_CLOUDY_STAGE:
        atmospheres_surface_cloudy[scenario_key] = make_atmosphere(
            planet,
            model_surface_cloudy,
            P,
            P_REF_BAR,
            R_p,
            T_input=T_input,
            X_input=X_input,
            surface_params=surface_params_mixed,
            cloud_params=cloud_params,
        )

surface_temperature_rows = []
for scenario_key in SCENARIO_KEYS:
    row = {"scenario": scenario_key}
    if scenario_key in atmospheres_surface_clear:
        row["T_surface_clear_K"] = float(
            atmospheres_surface_clear[scenario_key]["T_surf"]
        )
    if scenario_key in atmospheres_surface_cloudy:
        row["T_surface_cloudy_K"] = float(
            atmospheres_surface_cloudy[scenario_key]["T_surf"]
        )
    surface_temperature_rows.append(row)

display(pd.DataFrame(surface_temperature_rows))

## 9. Opacidades

La etapa nubosa requiere cargar además la base de aerosoles. Para minimizar ambigüedades se crean objetos de opacidad independientes para el modelo despejado y el nuboso.

In [ ]:
T_all = np.concatenate([temperatures[key] for key in SCENARIO_KEYS])
T_STEP = 10.0

T_FINE_MIN = max(100.0, np.floor(T_all.min() / T_STEP) * T_STEP - T_STEP)
T_FINE_MAX = min(400.0, np.ceil(T_all.max() / T_STEP) * T_STEP + T_STEP)
T_fine = np.arange(T_FINE_MIN, T_FINE_MAX + T_STEP, T_STEP)

LOG_P_FINE_MIN = -6.0
LOG_P_FINE_MAX = 0.0
LOG_P_FINE_STEP = 0.2
log_P_fine = np.arange(
    LOG_P_FINE_MIN,
    LOG_P_FINE_MAX + LOG_P_FINE_STEP,
    LOG_P_FINE_STEP,
)

print("T_fine [K]:", T_fine[0], "–", T_fine[-1])
print("log10(P/bar):", log_P_fine[0], "–", log_P_fine[-1])

In [ ]:
opac_clear = None
opac_cloudy = None

if RUN_GRAY_CONTROL or RUN_CLEAR_MIXED_SURFACE:
    opac_clear = read_opacities(
        model_surface_clear,
        wl,
        "opacity_sampling",
        T_fine,
        log_P_fine,
        opacity_database="Temperate",
    )

if RUN_CLOUDY_STAGE:
    opac_cloudy = read_opacities(
        model_surface_cloudy,
        wl,
        "opacity_sampling",
        T_fine,
        log_P_fine,
        opacity_database="Temperate",
    )

In [ ]:
physical_rows = []

for scenario_key in SCENARIO_KEYS:
    if RUN_GRAY_CONTROL:
        physical_rows.append({
            "scenario": scenario_key,
            "model": "gray_clear",
            "physical": check_atmosphere_physical(
                atmospheres_gray[scenario_key],
                opac_clear,
            ),
        })

    if RUN_CLEAR_MIXED_SURFACE:
        physical_rows.append({
            "scenario": scenario_key,
            "model": "mixed_clear",
            "physical": check_atmosphere_physical(
                atmospheres_surface_clear[scenario_key],
                opac_clear,
            ),
        })

    if RUN_CLOUDY_STAGE:
        physical_rows.append({
            "scenario": scenario_key,
            "model": "mixed_cloudy",
            "physical": check_atmosphere_physical(
                atmospheres_surface_cloudy[scenario_key],
                opac_cloudy,
            ),
        })

display(pd.DataFrame(physical_rows))

## 10. Espectros de emisión térmica directa

In [ ]:
spectra_gray = {}
spectra_surface_clear = {}
spectra_surface_cloudy = {}

for scenario_key in SCENARIO_KEYS:
    if RUN_GRAY_CONTROL:
        spectra_gray[scenario_key] = compute_spectrum(
            planet,
            star,
            model_gray,
            atmospheres_gray[scenario_key],
            opac_clear,
            wl,
            spectrum_type="direct_emission",
            use_photosphere_radius=USE_PHOTOSPHERE_RADIUS,
            save_spectrum=False,
        )

    if RUN_CLEAR_MIXED_SURFACE:
        spectra_surface_clear[scenario_key] = compute_spectrum(
            planet,
            star,
            model_surface_clear,
            atmospheres_surface_clear[scenario_key],
            opac_clear,
            wl,
            spectrum_type="direct_emission",
            use_photosphere_radius=USE_PHOTOSPHERE_RADIUS,
            save_spectrum=False,
        )

    if RUN_CLOUDY_STAGE:
        spectra_surface_cloudy[scenario_key] = compute_spectrum(
            planet,
            star,
            model_surface_cloudy,
            atmospheres_surface_cloudy[scenario_key],
            opac_cloudy,
            wl,
            spectrum_type="direct_emission",
            use_photosphere_radius=USE_PHOTOSPHERE_RADIUS,
            save_spectrum=False,
        )

for group_name, group in [
    ("gray", spectra_gray),
    ("mixed_clear", spectra_surface_clear),
    ("mixed_cloudy", spectra_surface_cloudy),
]:
    for key, spectrum in group.items():
        if np.any(~np.isfinite(spectrum)) or np.any(spectrum < 0):
            raise RuntimeError(
                f"Espectro no físico en {group_name}/{key}."
            )

print("Cálculo espectral completado.")

## 11. Unidades y rebineado

POSEIDON representa `Fp` en sus unidades de flujo nativas. Para análisis comparativo se muestran:

- flujo nativo;
- densidad de flujo \(F_\nu\) en nJy;
- espectros rebineados.

La conversión siguiente presupone que la salida nativa es \(F_\lambda\) por metro, conforme a la implementación de emisión directa utilizada por POSEIDON. Antes de exportar a LIFEsim, conviene comprobar esta convención contra la versión instalada.

In [ ]:
def flux_lambda_to_njy(flux_per_m, wavelength_um):
    wavelength_m = np.asarray(wavelength_um) * 1.0e-6
    flux_nu = np.asarray(flux_per_m) * wavelength_m**2 / c
    return flux_nu / 1.0e-26 * 1.0e9


def rebin_flux(wavelength_um, flux, resolving_power):
    wl_bin, flux_bin, _ = bin_spectrum(
        wavelength_um,
        flux,
        resolving_power,
    )
    return wl_bin, flux_bin


def normalized_shape(flux):
    flux = np.asarray(flux)
    scale = np.nanmax(flux)
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("No se puede normalizar el espectro.")
    return flux / scale

## 12. A0–A3 con superficie mixta despejada

In [ ]:
if RUN_CLEAR_MIXED_SURFACE:
    plt.figure(figsize=(11, 5.5))
    for key, info in SCENARIOS.items():
        wl_bin, flux_bin = rebin_flux(
            wl,
            spectra_surface_clear[key],
            R_PLOT,
        )
        plt.plot(
            wl_bin,
            flux_lambda_to_njy(flux_bin, wl_bin),
            label=f"{key}: {info['name']}",
        )

    plt.xlabel("Longitud de onda [µm]")
    plt.ylabel(r"$F_\nu$ [nJy]")
    plt.title(
        f"Superficie tipo Tierra, despejada — R={R_PLOT}"
    )
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.show()

## 13. Segunda etapa: superficie mixta con nubes parcheadas

In [ ]:
if RUN_CLOUDY_STAGE:
    plt.figure(figsize=(11, 5.5))
    for key, info in SCENARIOS.items():
        wl_bin, flux_bin = rebin_flux(
            wl,
            spectra_surface_cloudy[key],
            R_PLOT,
        )
        plt.plot(
            wl_bin,
            flux_lambda_to_njy(flux_bin, wl_bin),
            label=f"{key}: {info['name']}",
        )

    plt.xlabel("Longitud de onda [µm]")
    plt.ylabel(r"$F_\nu$ [nJy]")
    plt.title(
        f"Superficie tipo Tierra + {100 * F_CLOUD:.0f}% nube — R={R_PLOT}"
    )
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.show()

## 14. Efecto de la nube, escenario por escenario

La comparación mantiene constante la superficie mixta y cambia solamente la parametrización nubosa.

In [ ]:
if RUN_CLEAR_MIXED_SURFACE and RUN_CLOUDY_STAGE:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    axes = axes.ravel()

    for ax, key in zip(axes, SCENARIO_KEYS):
        wl_clear, clear_bin = rebin_flux(
            wl,
            spectra_surface_clear[key],
            R_PLOT,
        )
        wl_cloud, cloud_bin = rebin_flux(
            wl,
            spectra_surface_cloudy[key],
            R_PLOT,
        )

        clear_njy = flux_lambda_to_njy(clear_bin, wl_clear)
        cloud_njy = flux_lambda_to_njy(cloud_bin, wl_cloud)

        ax.plot(wl_clear, clear_njy, label="Clear")
        ax.plot(wl_cloud, cloud_njy, label="50% patchy cloud")
        ax.set_title(f"{key}: {SCENARIOS[key]['name']}")
        ax.set_ylabel(r"$F_\nu$ [nJy]")
        ax.grid(True, alpha=0.25)
        ax.legend()

    for ax in axes[-2:]:
        ax.set_xlabel("Longitud de onda [µm]")

    fig.suptitle("Efecto de las nubes sobre el espectro térmico")
    fig.tight_layout()
    plt.show()

In [ ]:
if RUN_CLEAR_MIXED_SURFACE and RUN_CLOUDY_STAGE:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    axes = axes.ravel()

    for ax, key in zip(axes, SCENARIO_KEYS):
        wl_bin, clear_bin = rebin_flux(
            wl,
            spectra_surface_clear[key],
            R_PLOT,
        )
        _, cloud_bin = rebin_flux(
            wl,
            spectra_surface_cloudy[key],
            R_PLOT,
        )

        clear_njy = flux_lambda_to_njy(clear_bin, wl_bin)
        cloud_njy = flux_lambda_to_njy(cloud_bin, wl_bin)
        delta = cloud_njy - clear_njy

        ax.plot(wl_bin, delta)
        ax.axhline(0.0, linewidth=1)
        ax.set_title(key)
        ax.set_ylabel("Cloudy − clear [nJy]")
        ax.grid(True, alpha=0.25)

    for ax in axes[-2:]:
        ax.set_xlabel("Longitud de onda [µm]")

    fig.suptitle("Cambio espectral inducido por el benchmark nuboso")
    fig.tight_layout()
    plt.show()

## 15. Efecto de introducir la superficie mixta

Esta sección compara el control gris y la superficie espectralmente mixta, ambos sin nubes.

In [ ]:
if RUN_GRAY_CONTROL and RUN_CLEAR_MIXED_SURFACE:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    axes = axes.ravel()

    for ax, key in zip(axes, SCENARIO_KEYS):
        wl_gray, gray_bin = rebin_flux(
            wl,
            spectra_gray[key],
            R_PLOT,
        )
        wl_mixed, mixed_bin = rebin_flux(
            wl,
            spectra_surface_clear[key],
            R_PLOT,
        )

        ax.plot(
            wl_gray,
            flux_lambda_to_njy(gray_bin, wl_gray),
            label="Gray",
        )
        ax.plot(
            wl_mixed,
            flux_lambda_to_njy(mixed_bin, wl_mixed),
            label="Earth-like mixture",
        )
        ax.set_title(key)
        ax.set_ylabel(r"$F_\nu$ [nJy]")
        ax.grid(True, alpha=0.25)
        ax.legend()

    for ax in axes[-2:]:
        ax.set_xlabel("Longitud de onda [µm]")

    fig.suptitle("Control gris frente a superficie mixta")
    fig.tight_layout()
    plt.show()

## 16. Diferencias A1–A3 respecto a A0

Esta es la señal astrofísica que posteriormente deberá competir con el ruido de LIFEsim.

In [ ]:
def plot_exofarm_differences(spectra_group, title):
    plt.figure(figsize=(11, 5))
    for key in ("A1", "A2", "A3"):
        wl_bin, scenario_bin = rebin_flux(
            wl,
            spectra_group[key],
            R_PLOT,
        )
        _, baseline_bin = rebin_flux(
            wl,
            spectra_group["A0"],
            R_PLOT,
        )

        delta_njy = flux_lambda_to_njy(
            scenario_bin - baseline_bin,
            wl_bin,
        )
        plt.plot(wl_bin, delta_njy, label=f"{key} − A0")

    plt.axhline(0.0, linewidth=1)
    plt.xlabel("Longitud de onda [µm]")
    plt.ylabel(r"$\Delta F_\nu$ [nJy]")
    plt.title(title)
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.show()


if RUN_CLEAR_MIXED_SURFACE:
    plot_exofarm_differences(
        spectra_surface_clear,
        "Diferencias ExoFarm — superficie mixta despejada",
    )

if RUN_CLOUDY_STAGE:
    plot_exofarm_differences(
        spectra_surface_cloudy,
        "Diferencias ExoFarm — superficie mixta con nubes",
    )

## 17. Comparación a resolución exploratoria de LIFE

El rebineado a \(R=50\) no sustituye la respuesta instrumental de LIFEsim; sirve para comprobar qué estructuras sobreviven a una resolución moderada.

In [ ]:
if RUN_CLEAR_MIXED_SURFACE and RUN_CLOUDY_STAGE:
    key = "A3"

    wl_r50, clear_r50 = rebin_flux(
        wl,
        spectra_surface_clear[key],
        R_LIFE_EXPLORATORY,
    )
    _, cloudy_r50 = rebin_flux(
        wl,
        spectra_surface_cloudy[key],
        R_LIFE_EXPLORATORY,
    )

    plt.figure(figsize=(11, 5))
    plt.plot(
        wl_r50,
        flux_lambda_to_njy(clear_r50, wl_r50),
        marker="o",
        label="A3 clear",
    )
    plt.plot(
        wl_r50,
        flux_lambda_to_njy(cloudy_r50, wl_r50),
        marker="s",
        label="A3 cloudy",
    )
    plt.xlabel("Longitud de onda [µm]")
    plt.ylabel(r"$F_\nu$ [nJy]")
    plt.title(f"A3 rebineado a R={R_LIFE_EXPLORATORY}")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.show()

## 18. Prueba leave-one-out para N₂O y NH₃

Se elimina una especie manteniendo fijo el perfil térmico y redistribuyendo la diferencia hacia N₂:

\[
\Delta F_{\rm mol}
=
F_{\rm full}-F_{\rm no\,mol}.
\]

Es un diagnóstico radiativo, no una nueva solución fotoquímica autoconsistente.

In [ ]:
def remove_species_and_close_with_n2(X_input, species):
    species_names = list(chemical_species_reference)
    if species not in species_names:
        raise ValueError(f"{species} no está en el modelo.")
    if "N2" not in species_names:
        raise ValueError("No se encontró N2 para cerrar la composición.")

    X_new = np.array(X_input, copy=True)
    species_index = species_names.index(species)
    n2_index = species_names.index("N2")

    removed = X_new[species_index].copy()
    X_new[species_index] = 0.0
    X_new[n2_index] += removed

    return X_new


def compute_leave_one_out(
    scenario_key,
    species,
    model,
    atmosphere_group,
    opac,
    surface_params,
    cloud_params_input=None,
):
    X_modified = remove_species_and_close_with_n2(
        compositions[scenario_key],
        species,
    )

    kwargs = dict(
        T_input=temperatures[scenario_key],
        X_input=X_modified,
        surface_params=surface_params,
    )
    if cloud_params_input is not None:
        kwargs["cloud_params"] = cloud_params_input

    atmosphere_modified = make_atmosphere(
        planet,
        model,
        P,
        P_REF_BAR,
        R_p,
        **kwargs,
    )

    return compute_spectrum(
        planet,
        star,
        model,
        atmosphere_modified,
        opac,
        wl,
        spectrum_type="direct_emission",
        use_photosphere_radius=USE_PHOTOSPHERE_RADIUS,
        save_spectrum=False,
    )

In [ ]:
leave_one_out = {}

if RUN_LEAVE_ONE_OUT:
    target_scenario = "A3"

    for species in ("N2O", "NH3"):
        leave_one_out[("clear", species)] = compute_leave_one_out(
            target_scenario,
            species,
            model_surface_clear,
            atmospheres_surface_clear,
            opac_clear,
            surface_params_mixed,
        )

        if RUN_CLOUDY_STAGE:
            leave_one_out[("cloudy", species)] = compute_leave_one_out(
                target_scenario,
                species,
                model_surface_cloudy,
                atmospheres_surface_cloudy,
                opac_cloudy,
                surface_params_mixed,
                cloud_params_input=cloud_params,
            )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

    for ax, species in zip(axes, ("N2O", "NH3")):
        wl_bin, full_clear = rebin_flux(
            wl,
            spectra_surface_clear[target_scenario],
            R_PLOT,
        )
        _, without_clear = rebin_flux(
            wl,
            leave_one_out[("clear", species)],
            R_PLOT,
        )

        delta_clear = flux_lambda_to_njy(
            full_clear - without_clear,
            wl_bin,
        )
        ax.plot(wl_bin, delta_clear, label="Clear")

        if RUN_CLOUDY_STAGE:
            _, full_cloudy = rebin_flux(
                wl,
                spectra_surface_cloudy[target_scenario],
                R_PLOT,
            )
            _, without_cloudy = rebin_flux(
                wl,
                leave_one_out[("cloudy", species)],
                R_PLOT,
            )

            delta_cloudy = flux_lambda_to_njy(
                full_cloudy - without_cloudy,
                wl_bin,
            )
            ax.plot(wl_bin, delta_cloudy, label="Cloudy")

        ax.axhline(0.0, linewidth=1)
        ax.set_title(species)
        ax.set_xlabel("Longitud de onda [µm]")
        ax.set_ylabel(r"$F_{\rm full}-F_{\rm no\,mol}$ [nJy]")
        ax.grid(True, alpha=0.25)
        ax.legend()

    fig.suptitle("Huella molecular leave-one-out en A3")
    fig.tight_layout()
    plt.show()

## 19. Métricas comparativas por banda

Estas métricas son amplitudes físicas del modelo, no significancias estadísticas.

In [ ]:
BANDS = {
    "N2O_7_9": (7.0, 9.0),
    "O3_9_6": (9.0, 10.2),
    "NH3_10_12": (10.0, 12.0),
    "CO2_15": (13.0, 17.0),
    "N2O_16_18_5": (16.0, 18.5),
}


def summarize_delta(reference, comparison, label):
    rows = []
    ref_njy = flux_lambda_to_njy(reference, wl)
    cmp_njy = flux_lambda_to_njy(comparison, wl)
    delta = cmp_njy - ref_njy

    for band, (left, right) in BANDS.items():
        mask = (wl >= left) & (wl <= right)
        rows.append({
            "comparison": label,
            "band": band,
            "wl_min_um": left,
            "wl_max_um": right,
            "mean_delta_nJy": float(np.mean(delta[mask])),
            "rms_delta_nJy": float(np.sqrt(np.mean(delta[mask] ** 2))),
            "max_abs_delta_nJy": float(np.max(np.abs(delta[mask]))),
        })

    return rows


metric_rows = []

if RUN_GRAY_CONTROL and RUN_CLEAR_MIXED_SURFACE:
    metric_rows.extend(
        summarize_delta(
            spectra_gray["A3"],
            spectra_surface_clear["A3"],
            "mixed_surface_minus_gray_A3",
        )
    )

if RUN_CLEAR_MIXED_SURFACE and RUN_CLOUDY_STAGE:
    metric_rows.extend(
        summarize_delta(
            spectra_surface_clear["A3"],
            spectra_surface_cloudy["A3"],
            "cloudy_minus_clear_A3",
        )
    )

if RUN_CLEAR_MIXED_SURFACE:
    metric_rows.extend(
        summarize_delta(
            spectra_surface_clear["A0"],
            spectra_surface_clear["A3"],
            "A3_minus_A0_clear",
        )
    )

if RUN_CLOUDY_STAGE:
    metric_rows.extend(
        summarize_delta(
            spectra_surface_cloudy["A0"],
            spectra_surface_cloudy["A3"],
            "A3_minus_A0_cloudy",
        )
    )

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

## 20. Exportación para LIFEsim

Se exportan espectros nativos y rebineados a \(R=50\). Los nombres distinguen:

- `gray_clear`;
- `earth_surface_clear`;
- `earth_surface_cloudy`.

Antes de importarlos en LIFEsim, confirma la unidad de flujo de la versión instalada de POSEIDON. LIFEsim debe recibir el espectro como flujo planetario absoluto, no como componente aditiva a un cuerpo negro.

In [ ]:
def export_group(group_name, spectra_group):
    exported = []

    for scenario_key, spectrum in spectra_group.items():
        native_df = pd.DataFrame({
            "wavelength_um": wl,
            "Fp_POSEIDON_native": spectrum,
            "flux_nJy_assuming_Flambda_per_m": flux_lambda_to_njy(
                spectrum,
                wl,
            ),
        })

        native_path = (
            OUTPUT_DIR
            / f"{scenario_key}_{group_name}_R{R_NATIVE}.csv"
        )
        native_df.to_csv(native_path, index=False)

        wl_r50, spectrum_r50 = rebin_flux(
            wl,
            spectrum,
            R_LIFE_EXPLORATORY,
        )
        r50_df = pd.DataFrame({
            "wavelength_um": wl_r50,
            "Fp_POSEIDON_native": spectrum_r50,
            "flux_nJy_assuming_Flambda_per_m": flux_lambda_to_njy(
                spectrum_r50,
                wl_r50,
            ),
        })

        r50_path = (
            OUTPUT_DIR
            / f"{scenario_key}_{group_name}_R{R_LIFE_EXPLORATORY}.csv"
        )
        r50_df.to_csv(r50_path, index=False)

        exported.extend([native_path, r50_path])

    return exported


exported_paths = []

if RUN_GRAY_CONTROL:
    exported_paths.extend(export_group("gray_clear", spectra_gray))

if RUN_CLEAR_MIXED_SURFACE:
    exported_paths.extend(
        export_group("earth_surface_clear", spectra_surface_clear)
    )

if RUN_CLOUDY_STAGE:
    exported_paths.extend(
        export_group("earth_surface_cloudy", spectra_surface_cloudy)
    )

metadata = {
    "POSEIDON_version": str(POSEIDON_VERSION),
    "planet": PLANET_NAME,
    "radius_R_earth": RADIUS_EARTH,
    "mass_M_earth": MASS_EARTH,
    "distance_pc": DISTANCE_PC,
    "surface_pressure_bar": P_SURF_BAR,
    "wavelength_range_um": [WL_MIN_UM, WL_MAX_UM],
    "native_resolving_power": R_NATIVE,
    "surface_components": SURFACE_COMPONENTS,
    "surface_fractions": SURFACE_FRACTIONS.tolist(),
    "cloud_benchmark": {
        "aerosol": CLOUD_AEROSOL,
        "fraction": F_CLOUD,
        "top_bar": P_CLOUD_TOP_BAR,
        "bottom_bar": P_CLOUD_BOTTOM_BAR,
        "mean_radius_um": R_CLOUD_UM,
        "log_X": LOG_X_CLOUD,
    },
    "profile_directory": str(PROFILES_DIR),
    "spectrum_type": "direct_emission",
}

metadata_path = OUTPUT_DIR / "thermal_forward_model_metadata.json"
metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("Archivos exportados:", len(exported_paths))
for path in exported_paths[:6]:
    print(" ", path)
if len(exported_paths) > 6:
    print(" ...")
print("Metadata:", metadata_path)

## 21. Interpretación correcta

### Qué representa la superficie mixta

Es una emisividad promedio de océano, bosque, nieve y arena bajo una única columna atmosférica 1D. No representa temperaturas regionales independientes.

### Qué representa la etapa nubosa

Es una prueba controlada de sensibilidad con una nube Mie parcheada. No es una climatología terrestre completa.

### Qué debemos mirar

1. Si \(\Delta F_{\rm superficie}\) es comparable a la huella de NH₃ cerca de 10–12 μm.
2. Si las nubes reducen o cambian de signo las diferencias A3–A0.
3. Si la banda larga de N₂O sobrevive al rebineado.
4. Si las conclusiones dependen más de la superficie/nube que del escenario químico.
5. Qué regiones espectrales conviene entregar a LIFEsim.

### Limitación crítica

Si los cuatro espectros superficiales del ejemplo HWO no cubren el MIR completo, deben sustituirse por datos de laboratorio adecuados. Extrapolarlos artificialmente produciría una superficie aparentemente sofisticada, pero físicamente indefendible.